In [ ]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install pandas tqdm scikit-learn javalang

In [ ]:
KAGGLE_INPUT_PATH = '...'
KAGGLE_OUTPUT_PATH = '.../data'

import os
os.makedirs(KAGGLE_OUTPUT_PATH, exist_ok=True)
print(f"Các tệp đầu vào nằm tại: {KAGGLE_INPUT_PATH}")
print(f"Các tệp đầu ra sẽ được lưu tại: {KAGGLE_OUTPUT_PATH}")

In [ ]:
import torch
print(torch.__version__)

In [ ]:
import sys

In [ ]:
import re
import javalang
from itertools import chain
from tqdm.notebook import tqdm

def parse_cfg(row):
    cfg, target = row['cfg'], row['node']
    node_list = []
    nodes = re.findall(r'(\d+)\s+\[shape=\w+,\s+label="([\s\S]+?)"];\n', cfg)

    for node in nodes:
        id_, text = eval(node[0]), node[1]
        try:
            tokens = [item.value for item in javalang.tokenizer.tokenize(text)]
        except Exception:
            # print(text)
            # exit(-1)
            return None 
            
        statement = ' '.join(tokens) if id_ != target else '_BOS_ ' + ' '.join(tokens) + ' _EOS_'
        node_list.append(statement)
        
    fwd_edges = re.findall(r'(\d+) -> (\d+) ;\n', cfg)
    back_edges = re.findall(r'(\d+) -> (\d+)\[style=dashed\];\n', cfg)
    fwd_edges = [[eval(edge[0]), eval(edge[1])] for edge in fwd_edges]
    back_edges = [[eval(edge[0]), eval(edge[1])] for edge in back_edges]

    for i, edge in enumerate(fwd_edges):
        begin, end = edge
        if begin > end:
            fwd_edges[i] = [end, begin]
            node_list[begin - 1], node_list[end - 1] = node_list[end - 1], node_list[begin - 1]
            for j, link in enumerate(fwd_edges):
                if j == i:
                    continue
                begin_idx, end_idx = -1, -1
                if begin in link:
                    begin_idx = link.index(begin)
                if end in link:
                    end_idx = link.index(end)
                if begin_idx != -1:
                    link[begin_idx] = end
                if end_idx != -1:
                    link[end_idx] = begin
                fwd_edges[j] = link
            for k, back in enumerate(back_edges):
                begin_idx, end_idx = -1, -1
                if begin in back:
                    begin_idx = back.index(begin)
                if end in back:
                    end_idx = back.index(end)
                if begin_idx != -1:
                    back[begin_idx] = end
                if end_idx != -1:
                    back[end_idx] = begin
                back_edges[k] = back
    if not back_edges:
        back_edges = [[1, 1]]
        
    final_target_index = -1
    for idx, node in enumerate(node_list):
        if node.startswith('_BOS_') and node.endswith('_EOS_'):
            final_target_index = idx
            break
            
    return node_list, fwd_edges, back_edges, final_target_index, row['label']

def run_preprocess(choices=['train', 'valid', 'test']):
    for choice in choices:
        input_file = KAGGLE_INPUT_PATH + f'{choice}.csv'
        output_file = KAGGLE_OUTPUT_PATH + f'{choice}.csv'
        print(f"\nBắt đầu tiền xử lý: {input_file}")

        df_raw = pd.read_csv(input_file)
        
        raw_data = df_raw[['cfg', 'node', 'label']].copy()
        raw_data.columns = ['cfg', 'node', 'label'] 

        items = raw_data.apply(parse_cfg, axis=1)
        items = items[items.notna()] 

        result = pd.DataFrame(data=list(items), columns=['nodes', 'forward', 'backward', 'target', 'label'])
        
        result.to_csv(output_file, index=False)
        print(f" Đã lưu tệp preprocess: {output_file} ({len(result):,} hàng)")

run_preprocess()

In [ ]:
df_train = pd.read_csv('.../data/train.csv')

# Xem 5 hàng đầu tiên
print(df_train.head())

In [ ]:
from ast import literal_eval

def safe_eval(value):
    """
    Convert chuỗi list thành list thật.
    Nếu value đã là list -> giữ nguyên.
    """
    if isinstance(value, list):
        return value
    try:
        return literal_eval(value)
    except:
        return value  


def annotate_api(row):
    # Chuyển nodes, forward, backward về dạng list thật
    nodes = safe_eval(row['nodes'])
    forward = safe_eval(row['forward'])
    backward = safe_eval(row['backward'])

    # ants = []
    # for node_text in nodes:
    #     annotation = 1 if re.findall(r'\.\s*\w+\s*\(', node_text) else 0
    #     ants.append(annotation)

    # return nodes, forward, backward, ants, row['label']

    role_labels = []
    for node_text in nodes:
        node_text_lower = node_text.lower()

        if any(kw in node_text for kw in ['BEGIN', 'EXIT']):
            role = 1
        elif any(kw in node_text_lower for kw in ['if ', 'else ', 'switch ', 'case ']):
            role = 2 
        elif any(kw in node_text_lower for kw in ['for ', 'while ', 'do ']):
            role = 3
        elif any(kw in node_text_lower for kw in ['return ', 'throw ', 'break ', 'continue ']):
            role = 4
        elif re.findall(r'\.\s*\w+\s*\(', node_text):  # api call tới 1 method
            role = 5
        else:
            role = 6 # lệnh gán/khai báo thường

        role_labels.append(role)

    return nodes, forward, backward, role_labels, row['label']

def run_annotation(choices=['train', 'valid', 'test']):
    annotation_type = 'api'

    for choice in choices:
        input_file = KAGGLE_OUTPUT_PATH + f'{choice}.csv'
        output_file = KAGGLE_OUTPUT_PATH + f'{choice}_{annotation_type}.csv'

        print(f"\n Đang xử lý: {input_file}")

        df_raw = pd.read_csv(input_file)

        # Gắn nhãn
        items = df_raw.apply(annotate_api, axis=1)

        result = pd.DataFrame(
            data=list(items),
            columns=['nodes', 'forward', 'backward', 'types', 'label']
        )

        result.to_csv(output_file, index=False, encoding='utf-8')
        print(f" Đã lưu: {output_file}")

run_annotation()

In [ ]:
!pip install tokenizers

In [ ]:
import json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Cấu hình
TRAIN_PATH = ".../train_api.csv"
VALID_PATH = ".../valid_api.csv"
TEST_PATH  = ".../test_api.csv"
OUT_DIR = ".../data"

VOCAB_SIZE = 50000 
MAX_TOKEN_PER_NODE = 20 

def train_bpe_tokenizer(csv_path, vocab_size=VOCAB_SIZE):
    print(f"🔹 Training BPE Tokenizer from: {csv_path}")
    
    # 1. Khởi tạo BPE Model
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace() # Tách khoảng trắng trước khi tách subword

    # 2. Chuẩn bị dữ liệu huấn luyện (lấy text từ tất cả các nodes)
    df = pd.read_csv(csv_path)
    all_texts = []
    for _, row in df.iterrows():
        nodes = eval(row["nodes"])
        all_texts.extend(nodes)

    # 3. Huấn luyện
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<pad>", "<unk>", "<s>", "</s>", "_BOS_", "_EOS_"] # Thêm các token đặc biệt
    )
    tokenizer.train_from_iterator(all_texts, trainer=trainer)
    
    # Đảm bảo <pad> luôn là ID 0
    return tokenizer

def convert_csv_to_subword_ids(src_path, dst_path, tokenizer, max_token=MAX_TOKEN_PER_NODE):
    print(f"Converting {src_path} → {dst_path}")
    df = pd.read_csv(src_path)
    new_nodes = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        nodes = eval(row["nodes"])
        node_ids = []

        for node in nodes:
            # Encode text sang Subword IDs
            output = tokenizer.encode(node)
            ids = output.ids[:max_token] # Cắt độ dài theo max_token
            node_ids.append(ids)

        new_nodes.append(node_ids)

    df["nodes"] = new_nodes
    df.to_csv(dst_path, index=False)
    print(f"Saved: {dst_path}")

if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)

    # Huấn luyện bộ mã hóa BPE trên tập Train
    bpe_tokenizer = train_bpe_tokenizer(TRAIN_PATH)

    # Lưu bộ mã hóa để tái sử dụng
    tokenizer_path = os.path.join(OUT_DIR, "bpe_tokenizer.json")
    bpe_tokenizer.save(tokenizer_path)
    print(f"Tokenizer saved to {tokenizer_path}")

    # Chuyển đổi dữ liệu sang ID bằng BPE
    convert_csv_to_subword_ids(TRAIN_PATH, os.path.join(OUT_DIR, "train_api_id.csv"), bpe_tokenizer)
    convert_csv_to_subword_ids(VALID_PATH, os.path.join(OUT_DIR, "valid_api_id.csv"), bpe_tokenizer)
    convert_csv_to_subword_ids(TEST_PATH,  os.path.join(OUT_DIR, "test_api_id.csv"), bpe_tokenizer)

In [ ]:
!pip install torch-geometric

In [ ]:
class CFG:
    vocab_size = 50000
    batch_size = 64
    hidden_dim = 128
    max_node = 300 # or '150' / '200' / '250' / '350' / '400'
    max_token = 20 # or '10' / '30'/ '40' / '50'
    learning_rate = 0.0005
    epoch = 15
    model_type = 'GAT'  # or 'GAT' / 'GCN' / 'Transformer'
    gat_heads = 4 # or 2 / 8 / 16 / 32

    # Device settings
    cpu = False     
    gpu = 0         
    num_workers = 2       
    pin_memory = True    

    save_model = False

cfg = CFG()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class APIDataset(Dataset):
    def __init__(self, csv_path, max_node, max_token):
        df = pd.read_csv(csv_path)

        self.nodes = [literal_eval(x) for x in df["nodes"]]
        self.f_edges = [literal_eval(x) for x in df["forward"]]
        self.b_edges = [literal_eval(x) for x in df["backward"]]
        self.types = [literal_eval(x) for x in df["types"]]
        self.labels = df["label"].tolist()

        self.max_node = max_node
        self.max_token = max_token

        print(f"Loaded {len(self.labels)} samples from {csv_path}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.nodes[idx],
            self.f_edges[idx],
            self.b_edges[idx],
            self.types[idx],
            self.labels[idx]
        )


def collate_fn(batch, max_node, max_token):
    B = len(batch)

    node_tensor = torch.zeros((B, max_node, max_token), dtype=torch.long)
    type_tensor = torch.zeros((B, max_node), dtype=torch.long)
    label_tensor = torch.tensor([b[4] for b in batch], dtype=torch.long)

    f_edge_list = []
    b_edge_list = []

    for i, (nodes, f_edges, b_edges, types, _) in enumerate(batch):
        n_nodes = min(len(nodes), max_node)
        offset = i * max_node # chỉ số bắt đầu của sample thứ i

        for j in range(n_nodes):
            tokens = nodes[j][:max_token]
            node_tensor[i, j, :len(tokens)] = torch.tensor(tokens)

        type_tensor[i, :n_nodes] = torch.tensor(types[:n_nodes])

        # vector hóa f_edges và b_edges = cách dịch chuyển chỉ số -> gộp batch lớn
        for u, v in f_edges:
            if u < max_node and v < max_node:
                f_edge_list.append([u + offset, v + offset])

        for u, v in b_edges:
            if u < max_node and v < max_node:
                b_edge_list.append([u + offset, v + offset])

        # chuyển sang tensor [2; E]
        edge_index_f = torch.tensor(f_edge_list, dtype=torch.long).t().contiguous() if f_edge_list else torch.empty((2, 0), dtype=torch.long)
        edge_index_b = torch.tensor(b_edge_list, dtype=torch.long).t().contiguous() if b_edge_list else torch.empty((2, 0), dtype=torch.long)
        
    return node_tensor, type_tensor, edge_index_f, edge_index_b, label_tensor


def get_loaders(cfg, train_path, valid_path):
    
    train_ds = APIDataset(train_path, cfg.max_node, cfg.max_token)
    valid_ds = APIDataset(valid_path, cfg.max_node, cfg.max_token)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, cfg.max_node, cfg.max_token),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, cfg.max_node, cfg.max_token),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )

    return train_loader, valid_loader

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CFGNN_NoGAT(nn.Module):
    def __init__(self, cfg, num_classes=2):
        super().__init__()
        D = cfg.hidden_dim

        self.token_emb = nn.Embedding(cfg.vocab_size, D, padding_idx=0)
        self.type_emb = nn.Embedding(7, D)
        self.dropout = nn.Dropout(p=0.3) 

        # BiLSTM
        self.node_lstm = nn.LSTM(
            input_size=D,
            hidden_size=D // 2,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Linear(D, 1)
        self.classifier = nn.Linear(D, num_classes)

    def forward(self, nodes, types, edge_index_f, edge_index_b):
        B, N, T = nodes.size()

        # 1. Token -> Node embedding
        token_emb = self.token_emb(nodes)
        mask = (nodes != 0).float().unsqueeze(-1)
        node_emb = (token_emb * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1)

        # 2. Kết hợp thông tin loại nút
        node_emb = node_emb + self.type_emb(types)
        node_emb = self.dropout(node_emb)

        # 3. Dữ liệu đi thẳng từ node_emb vào BiLSTM
        
        node_out, _ = self.node_lstm(node_emb)
        node_out = self.dropout(node_out)

        # 4. Attention pooling và Classifier
        attn_score = self.attn(node_out).squeeze(-1)
        node_mask = (types != 0) 
        attn_score = attn_score.masked_fill(~node_mask, -1e9)
        attn_weight = F.softmax(attn_score, dim=1)

        graph_repr = torch.sum(node_out * attn_weight.unsqueeze(-1), dim=1)

        return self.classifier(graph_repr)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

from sklearn.metrics import precision_score, recall_score, f1_score

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="Train", leave=False)
    for nodes, types, f_edges, b_edges, labels in pbar:
        nodes = nodes.to(device)
        types = types.to(device)
        f_edges = f_edges.to(device) 
        b_edges = b_edges.to(device) 
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(nodes, types, f_edges, b_edges)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for nodes, types, f_edges, b_edges, labels in tqdm(loader, desc="Valid", leave=False):
            nodes = nodes.to(device)
            types = types.to(device)
            f_edges = f_edges.to(device) 
            b_edges = b_edges.to(device) 
            labels = labels.to(device)

            logits = model(nodes, types, f_edges, b_edges)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = total_loss / len(loader)
    val_acc = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()
    val_precision = precision_score(all_labels, all_preds, zero_division=0)
    val_recall = recall_score(all_labels, all_preds, zero_division=0)
    val_f1 = f1_score(all_labels, all_preds, zero_division=0)

    return val_loss, val_acc, val_precision, val_recall, val_f1


def main():
    device = torch.device("cuda" if torch.cuda.is_available() and not cfg.cpu else "cpu")
    print(" Using device:", device)

    train_loader, valid_loader = get_loaders(
        cfg,
        KAGGLE_INPUT_PATH + 'trainpath',
        KAGGLE_INPUT_PATH + 'validpath'
    )

    model = CFGNN_NoGAT(cfg).to(device)
    optimizer = optim.Adam(
        model.parameters(), 
        lr=cfg.learning_rate,
        weight_decay=1e-4
    )

    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, 6.0]).to(device)
    )

    best_val_f1 = 0.0
    best_model_path = ".../best_model.pt"

    patience = 5
    no_improve = 0

    print(" Start training...")
    for epoch in range(cfg.epoch):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        va_loss, va_acc, va_prec, va_rec, va_f1 = eval_one_epoch(
            model, valid_loader, criterion, device
        )

        print(
            f"Epoch {epoch+1}/{cfg.epoch} | "
            f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} || "
            f"Val Loss={va_loss:.4f} Acc={va_acc:.4f} "
            f"P={va_prec:.4f} R={va_rec:.4f} F1={va_f1:.4f}"
        )

        #  CHECK IMPROVEMENT
        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            no_improve = 0

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": va_f1
            }, best_model_path)

            print(f" Saved best model (Val F1={va_f1:.4f})")

        else:
            no_improve += 1
            print(f" No improvement ({no_improve}/{patience})")

        # EARLY STOP
        if no_improve >= patience:
            print(
                f"\n Early stopping at epoch {epoch+1} "
                f"(Best Val F1={best_val_f1:.4f})"
            )
            break

    print(f"\n Best Validation F1-score: {best_val_f1:.4f}")
    
main()

In [ ]:
def load_best_model(model, path, device):
    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False  
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(
        f" Loaded best model from epoch {checkpoint['epoch']} "
        f"(Val F1={checkpoint['val_f1']:.4f})"
    )

    return model

In [ ]:
def get_test_loader(cfg, test_path):
    test_ds = APIDataset(test_path, cfg.max_node, cfg.max_token)

    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, cfg.max_node, cfg.max_token),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory
    )
    return test_loader

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix
)

def test_model(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for nodes, types, f_edges, b_edges, labels in tqdm(loader, desc="Test"):
            nodes = nodes.to(device)
            types = types.to(device)
            f_edges = f_edges.to(device) 
            b_edges = b_edges.to(device) 
            labels = labels.to(device)

            logits = model(nodes, types, f_edges, b_edges)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    # matrix
    cm = confusion_matrix(all_labels, all_preds)

    print("\n📊 TEST RESULTS")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    return acc, prec, rec, f1, cm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test loader
test_loader = get_test_loader(
    cfg,
    KAGGLE_INPUT_PATH + 'testpath'
)

# Load model
model = CFGNN_NoGAT(cfg).to(device)
model = load_best_model(
    model,
    ".../best_model.pt",
    device
)

# Run test
test_model(model, test_loader, device)